# Experiment 02

# Google Earth Engine Data Preparation

## Objectives

This notebook prepares the cleaned Indian MOREPOC dataset for satellite reflectance extraction using Google Earth Engine.

The workflow includes:

- Loading the cleaned dataset
- Validating geographic coordinates
- Converting samples into Earth Engine Features
- Building an Earth Engine FeatureCollection
- Preparing data for Landsat scene assignment

The output of this notebook will be an Earth Engine FeatureCollection that serves as the input for Landsat surface reflectance extraction.

---

### Input

Indian_MOREPOC_GEE_Input.csv

---

### Output

Earth Engine FeatureCollection

In [1]:
# ============================================================
# Section 2 : Import Libraries
# ============================================================

import os
import warnings

import numpy as np
import pandas as pd

import ee
import geemap

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

print("="*70)
print("Libraries Imported Successfully")
print("="*70)

Libraries Imported Successfully


### Interpretation

This section imports all libraries required for data manipulation, visualization, and communication with Google Earth Engine.

NumPy and Pandas are used for preprocessing, while the Earth Engine Python API and geemap provide access to satellite imagery and cloud-based geospatial computation.

In [2]:
# ============================================================
# Section 3 : Initialize Google Earth Engine
# ============================================================

ee.Initialize(project="oc-flux")

Map = geemap.Map()

print("="*70)
print("Google Earth Engine Initialized Successfully")
print("="*70)

Google Earth Engine Initialized Successfully


### Interpretation

The Google Earth Engine API has been successfully initialized using the project associated with this research.

All subsequent satellite image processing will be executed on Google's cloud platform, eliminating the need for local storage or computational resources.

In [3]:
# ============================================================
# Section 4 : Load GEE Dataset
# ============================================================

gee_df = pd.read_csv(
    "Indian_MOREPOC_GEE_Input.csv",
    parse_dates=["sample_date"]
)

print("="*70)
print("Dataset Loaded")
print("="*70)

print("Shape :", gee_df.shape)

display(gee_df.head())

Dataset Loaded
Shape : (27, 29)


,bas_id,riv_id,country,cont,code,time_m,time_d,time_y,lat,lon,type_spm,fra_spm,conc_spm,conc_poc,perc_poc,perc_poc_1sd,d13C_poc,d13C_1sd,D14C_poc,D14C_1sd,age_14C,F14C,perc_tn,cn_ratio,alsi_ratio,ref,sample_date,coordinate_status,sample_id
0,Brahmaputra,Brahmaputra,India,Asia,BR 3 Brahmaputra-Guwahati,10.0,25.0,1999.0,26.19,91.74,PSD,Bulk,800.0,3.9,0.49,0.03,-24.5,0.25,NaN,NaN,NaN,NaN,NaN,NaN,0.27,Galy et al. (2008),1999-10-25,Available,IND_0001
1,Brahmaputra,Brahmaputra,India,Asia,BR 4 Brahmaputra-Guwahati,10.0,25.0,1999.0,26.19,91.74,PSD,Bulk,1800.0,5.4,0.30,0.03,-24.7,0.25,NaN,NaN,NaN,NaN,NaN,NaN,0.23,Galy et al. (2008),1999-10-25,Available,IND_0002
2,Brahmaputra,Brahmaputra,India,Asia,BR 6 Brahmaputra-Guwahati,10.0,25.0,1999.0,26.19,91.74,PSD,Bulk,800.0,5.9,0.74,0.03,-24.7,0.25,NaN,NaN,NaN,NaN,NaN,NaN,0.31,Galy et al. (2008),1999-10-25,Available,IND_0003
3,Brahmaputra,Brahmaputra,India,Asia,BR 7 Brahmaputra-Guwahati,10.0,25.0,1999.0,26.19,91.74,PSD,Bulk,2500.0,5.3,0.21,0.03,-25.3,0.25,NaN,NaN,NaN,NaN,NaN,NaN,0.21,Galy et al. (2008),1999-10-25,Available,IND_0004
4,Brahmaputra,Brahmaputra,India,Asia,BR 8 Brahmaputra-Guwahati,10.0,25.0,1999.0,26.19,91.74,PSD,Bulk,1200.0,5.5,0.46,0.03,-25.5,0.25,NaN,NaN,NaN,NaN,NaN,NaN,0.25,Galy et al. (2008),1999-10-25,Available,IND_0005


### Interpretation

The dataset loaded in this section contains only observations with valid geographic coordinates.

These samples are suitable for spatial querying within Google Earth Engine and will be used for Landsat image assignment and reflectance extraction.

# Section 5 : Dataset Validation

## Objective

Before creating an Earth Engine FeatureCollection, the dataset must be validated to ensure that all records satisfy the requirements for spatial processing.

The validation includes:

- Dataset dimensions
- Missing values
- Coordinate ranges
- Duplicate sample identifiers
- Duplicate coordinates
- Date format
- River distribution
- Summary statistics of the target variable (POC)

Performing these checks before Earth Engine processing prevents errors during image extraction and improves reproducibility.

In [4]:
# ============================================================
# Section 5.1 : Dataset Overview
# ============================================================

print("="*70)
print("Dataset Overview")
print("="*70)

print("Rows :", len(gee_df))
print("Columns :", len(gee_df.columns))

print("\nColumn Names")

for col in gee_df.columns:
    print("-", col)

Dataset Overview
Rows : 27
Columns : 29

Column Names
- bas_id
- riv_id
- country
- cont
- code
- time_m
- time_d
- time_y
- lat
- lon
- type_spm
- fra_spm
- conc_spm
- conc_poc
- perc_poc
- perc_poc_1sd
- d13C_poc
- d13C_1sd
- D14C_poc
- D14C_1sd
- age_14C
- F14C
- perc_tn
- cn_ratio
- alsi_ratio
- ref
- sample_date
- coordinate_status
- sample_id


In [5]:
# ============================================================
# Section 5.2 : Missing Values
# ============================================================

missing = gee_df.isnull().sum()

missing = pd.DataFrame({

    "Missing Values": missing,

    "Percentage (%)":
        round(missing / len(gee_df) * 100,2)

})

display(missing)

,Missing Values,Percentage (%)
bas_id,0,0.00
riv_id,0,0.00
country,0,0.00
cont,0,0.00
code,0,0.00
time_m,0,0.00
time_d,0,0.00
time_y,0,0.00
lat,0,0.00
lon,0,0.00


In [6]:
# ============================================================
# Section 5.3 : Coordinate Validation
# ============================================================

print("="*70)
print("Coordinate Validation")
print("="*70)

print("Latitude Range")

print(
    gee_df["lat"].min(),
    "to",
    gee_df["lat"].max()
)

print()

print("Longitude Range")

print(
    gee_df["lon"].min(),
    "to",
    gee_df["lon"].max()
)

Coordinate Validation
Latitude Range
25.3 to 27.74

Longitude Range
83.01 to 91.74


In [7]:
# ============================================================
# Section 5.4 : Duplicate Sample IDs
# ============================================================

duplicates = gee_df["sample_id"].duplicated().sum()

print("="*70)
print("Duplicate Sample IDs")
print("="*70)

print(duplicates)

Duplicate Sample IDs
0


In [8]:
# ============================================================
# Section 5.5 : Duplicate Coordinates
# ============================================================

duplicate_locations = gee_df.duplicated(

    subset=["lat","lon","sample_date"]

).sum()

print("="*70)
print("Duplicate Coordinate-Date Pairs")
print("="*70)

print(duplicate_locations)

Duplicate Coordinate-Date Pairs
16


In [9]:
# ============================================================
# Section 5.6 : River Distribution
# ============================================================

river_summary = (

    gee_df["riv_id"]

    .value_counts()

    .reset_index()

)

river_summary.columns = [

    "River",

    "Samples"

]

display(river_summary)

,River,Samples
0,Ganges,20
1,Brahmaputra,7


In [10]:
# ============================================================
# Section 5.7 : Date Validation
# ============================================================

print("="*70)
print("Date Validation")
print("="*70)

print("Earliest Sample")

print(
    gee_df["sample_date"].min()
)

print()

print("Latest Sample")

print(
    gee_df["sample_date"].max()
)

Date Validation
Earliest Sample
1997-03-15 00:00:00

Latest Sample
2005-07-13 00:00:00


In [11]:
# ============================================================
# Section 5.8 : POC Summary
# ============================================================

display(

gee_df["conc_poc"].describe()

)

count    27.000000
mean      6.077778
std       3.515169
min       0.100000
25%       4.900000
50%       5.800000
75%       6.900000
max      14.500000
Name: conc_poc, dtype: float64

### Interpretation

The validation confirms that the Google Earth Engine input dataset is internally consistent and suitable for remote sensing analysis.

All observations contain valid geographic coordinates, standardized sampling dates, and unique sample identifiers. Duplicate sample identifiers are absent, ensuring that each field observation represents a unique sampling event.

The dataset consists of coordinate-complete observations from the Ganges and Brahmaputra river systems, which will be used for Landsat surface reflectance extraction. Summary statistics of particulate organic carbon (POC) indicate a realistic range of concentrations suitable for subsequent modelling.

Successful completion of this validation step confirms that the dataset is ready for conversion into an Earth Engine FeatureCollection.

# Section 6 : Create Earth Engine FeatureCollection

## Objective

Google Earth Engine operates on server-side geospatial objects rather than Pandas DataFrames.

In this section, the cleaned MOREPOC observations are converted into Earth Engine Features, where each observation consists of:

- Point geometry (latitude and longitude)
- Sample metadata
- Sampling date
- POC concentration
- Unique sample identifier

These individual Features are then combined into an Earth Engine FeatureCollection, which becomes the primary input for all subsequent Landsat image assignment and reflectance extraction.

In [12]:
# ============================================================
# Section 6.1 : Create Earth Engine Features
# ============================================================

features = []

for _, row in gee_df.iterrows():

    geometry = ee.Geometry.Point(
        [row["lon"], row["lat"]]
    )

    feature = ee.Feature(
        geometry,
        {
            "sample_id": row["sample_id"],
            "riv_id": row["riv_id"],
            "country": row["country"],
            "sample_date": row["sample_date"].strftime("%Y-%m-%d"),
            "conc_poc": float(row["conc_poc"])
        }
    )

    features.append(feature)

print("="*70)
print("Earth Engine Features Created")
print("="*70)

print("Total Features :", len(features))

Earth Engine Features Created
Total Features : 27


In [13]:
# ============================================================
# Section 6.2 : FeatureCollection
# ============================================================

fc = ee.FeatureCollection(features)

print("="*70)
print("FeatureCollection Created")
print("="*70)

print(

fc.size().getInfo()

)

FeatureCollection Created
27


In [14]:
# ============================================================
# Section 6.3 : First Feature
# ============================================================

first_feature = fc.first().getInfo()

print("="*70)
print("First Feature")
print("="*70)

print(first_feature)

First Feature
{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [91.74, 26.19]}, 'id': '0', 'properties': {'conc_poc': 3.9, 'country': 'India', 'riv_id': 'Brahmaputra ', 'sample_date': '1999-10-25', 'sample_id': 'IND_0001'}}


In [15]:
# ============================================================
# Section 6.4 : Visualize Sampling Points
# ============================================================

Map = geemap.Map()

Map.centerObject(fc, 5)

Map.addLayer(
    fc,
    {
        "color": "red"
    },
    "Sampling Locations"
)

Map

Map(center=[25.982832753845763, 86.22223959274547], controls=(WidgetControl(options=['position', 'transparent_…

In [16]:
# ============================================================
# Section 6.5 : Feature Properties
# ============================================================

print("="*70)
print("Feature Properties")
print("="*70)

print(

fc.first().propertyNames().getInfo()

)

Feature Properties
['system:index', 'country', 'sample_id', 'sample_date', 'conc_poc', 'riv_id']


### Interpretation

The cleaned sampling observations have been successfully converted into an Earth Engine FeatureCollection.

Each feature consists of a geographic point representing the sampling location and a set of descriptive attributes, including the sample identifier, river name, sampling date, and observed particulate organic carbon (POC) concentration.

The FeatureCollection forms the foundation for all subsequent Google Earth Engine analyses, including Landsat scene assignment and surface reflectance extraction. Converting the tabular dataset into a server-side geospatial object enables efficient cloud-based processing of satellite imagery.

# Section 7.1 : Load Landsat Surface Reflectance Collections

## Objective

Different Landsat missions use different band names. To create a unified extraction workflow, all Landsat collections are loaded first and their spectral bands are harmonized to common names.

The common band names used throughout this project are:

- Blue
- Green
- Red
- NIR
- SWIR1
- SWIR2

This harmonization allows the same extraction function to work regardless of whether the nearest image originates from Landsat 5, Landsat 7, Landsat 8, or Landsat 9.

In [17]:
# ============================================================
# Section 7.1 : Load Landsat Collections
# ============================================================

# Landsat 5
LS5 = ee.ImageCollection(
    "LANDSAT/LT05/C02/T1_L2"
)

# Landsat 7
LS7 = ee.ImageCollection(
    "LANDSAT/LE07/C02/T1_L2"
)

# Landsat 8
LS8 = ee.ImageCollection(
    "LANDSAT/LC08/C02/T1_L2"
)

# Landsat 9
LS9 = ee.ImageCollection(
    "LANDSAT/LC09/C02/T1_L2"
)

print("="*70)
print("Landsat Collections Loaded")
print("="*70)

print("LS5 :", LS5.size().getInfo())
print("LS7 :", LS7.size().getInfo())
print("LS8 :", LS8.size().getInfo())
print("LS9 :", LS9.size().getInfo())

Landsat Collections Loaded
LS5 : 1876269
LS7 : 2580169
LS8 : 2048380
LS9 : 729077


In [18]:
# ============================================================
# Section 7.2 : Cloud Mask
# ============================================================

def mask_clouds(image):

    qa = image.select("QA_PIXEL")

    mask = (
        qa.bitwiseAnd(1 << 3).eq(0)  # Cloud
        .And(
            qa.bitwiseAnd(1 << 4).eq(0)  # Cloud Shadow
        )
    )

    return image.updateMask(mask)

In [19]:
# ============================================================
# Section 7.3 : Harmonize Bands
# ============================================================

def harmonize(image):

    spacecraft = image.get("SPACECRAFT_ID")

    spacecraft = ee.String(spacecraft)

    is_old = spacecraft.compareTo("LANDSAT_5").eq(0).Or(
        spacecraft.compareTo("LANDSAT_7").eq(0)
    )

    old = image.select(
        ["SR_B1","SR_B2","SR_B3","SR_B4","SR_B5","SR_B7"],
        ["Blue","Green","Red","NIR","SWIR1","SWIR2"]
    )

    new = image.select(
        ["SR_B2","SR_B3","SR_B4","SR_B5","SR_B6","SR_B7"],
        ["Blue","Green","Red","NIR","SWIR1","SWIR2"]
    )

    return ee.Image(
        ee.Algorithms.If(
            is_old,
            old,
            new
        )
    ).copyProperties(
        image,
        image.propertyNames()
    )

In [20]:
# ============================================================
# Section 7.4 : Unified Landsat Collection
# ============================================================

landsat = (

    LS5
    .merge(LS7)
    .merge(LS8)
    .merge(LS9)

)

print("="*70)
print("Unified Landsat Collection")
print("="*70)

print(

landsat.size().getInfo()

)

Unified Landsat Collection
7233895


### Interpretation

All four Landsat missions have been successfully combined into a single harmonized image collection.

A cloud masking function was defined to remove cloud and cloud-shadow contaminated pixels using the QA_PIXEL quality band.

The harmonization function standardizes the spectral band names across Landsat missions, ensuring that subsequent reflectance extraction can be performed using one consistent workflow without mission-specific code.

This unified collection forms the basis for identifying the nearest cloud-free Landsat observation for each sampling location.

# Section 7.5 : Nearest Landsat Scene Search

## Objective

Landsat satellites revisit the same location approximately every 16 days, therefore an image is rarely available on the exact sampling date.

Instead of requiring an exact match, this workflow searches within ±30 days of the sampling date and selects the observation with the smallest temporal difference.

This approach maximizes the probability of obtaining cloud-free imagery while minimizing temporal mismatch between field observations and satellite measurements.

In [21]:
# ============================================================
# Section 7.5 : Find Nearest Landsat Scene
# ============================================================

def find_nearest_scene(feature):

    sample_date = ee.Date(feature.get("sample_date"))
    geometry = feature.geometry()

    collection = (
        landsat
        .filterBounds(geometry)
        .filterDate(
            sample_date.advance(-60, "day"),
            sample_date.advance(60, "day")
        )
        .map(mask_clouds)
        .map(harmonize)
    )

    def add_difference(image):

        difference = (
            ee.Date(image.get("system:time_start"))
            .difference(sample_date, "day")
            .abs()
        )

        return image.set("date_difference", difference)

    collection = collection.map(add_difference)

    nearest = collection.sort("date_difference").first()

    return ee.Feature(feature).set({

        "scene_id":
            nearest.get("system:index"),

        "scene_date":
            ee.Date(
                nearest.get("system:time_start")
            ).format("YYYY-MM-dd"),

        "sensor":
            nearest.get("SPACECRAFT_ID"),

        "cloud_cover":
            nearest.get("CLOUD_COVER"),

        "date_difference":
            nearest.get("date_difference")

    })

In [22]:
# ============================================================
# Section 7.6 : Test
# ============================================================

test_feature = ee.Feature(fc.first())

test = find_nearest_scene(test_feature)

print("="*70)
print("Nearest Scene")
print("="*70)

print(

test.toDictionary().getInfo()

)

Nearest Scene
{'cloud_cover': 34, 'conc_poc': 3.9, 'country': 'India', 'date_difference': 0.8372787615740741, 'riv_id': 'Brahmaputra ', 'sample_date': '1999-10-25', 'sample_id': 'IND_0001', 'scene_date': '1999-10-24', 'scene_id': '1_1_1_LT05_136042_19991024', 'sensor': 'LANDSAT_5'}


# Section 8 : Landsat Availability Check

## Objective

Before assigning Landsat scenes, it is important to verify that each sampling location has at least one available Landsat image within the search window.

Rather than allowing the extraction function to fail when no imagery exists, the dataset is first divided into:

- Samples with at least one Landsat image
- Samples without any available Landsat imagery

Only the valid observations will proceed to scene assignment.

In [23]:
# ============================================================
# Section 8.1 : Landsat Availability Function
# ============================================================

def has_landsat(feature):

    sample_date = ee.Date(feature.get("sample_date"))

    geometry = feature.geometry()

    image_count = (

        landsat

        .filterBounds(geometry)

        .filterDate(

            sample_date.advance(-60, "day"),

            sample_date.advance(60, "day")

        )

        .size()

    )

    return feature.set("image_count", image_count)

In [24]:
# ============================================================
# Section 8.2 : Apply Availability Function
# ============================================================

availability_fc = fc.map(has_landsat)

print("="*70)
print("Availability Check Completed")
print("="*70)

print(

availability_fc.size().getInfo()

)

Availability Check Completed
27


In [25]:
# ============================================================
# Section 8.3 : Split Dataset
# ============================================================

valid_fc = availability_fc.filter(

    ee.Filter.gt("image_count", 0)

)

missing_fc = availability_fc.filter(

    ee.Filter.eq("image_count", 0)

)

print("="*70)
print("Available Samples")
print("="*70)

print(valid_fc.size().getInfo())

print()

print("="*70)
print("Missing Samples")
print("="*70)

print(missing_fc.size().getInfo())

Available Samples
27

Missing Samples
0


In [26]:
# ============================================================
# Section 8.4 : Missing Samples
# ============================================================

missing_features = missing_fc.getInfo()["features"]

missing_records = []

for feature in missing_features:

    missing_records.append(

        feature["properties"]

    )

missing_df = pd.DataFrame(missing_records)

display(missing_df)

""


In [27]:
# ============================================================
# Section 8.5 : Valid Samples
# ============================================================

valid_features = valid_fc.getInfo()["features"]

valid_records = []

for feature in valid_features:

    valid_records.append(

        feature["properties"]

    )

valid_df = pd.DataFrame(valid_records)

display(valid_df.head())

,conc_poc,country,image_count,riv_id,sample_date,sample_id
0,3.9,India,19,Brahmaputra,1999-10-25,IND_0001
1,5.4,India,19,Brahmaputra,1999-10-25,IND_0002
2,5.9,India,19,Brahmaputra,1999-10-25,IND_0003
3,5.3,India,19,Brahmaputra,1999-10-25,IND_0004
4,5.5,India,19,Brahmaputra,1999-10-25,IND_0005


In [28]:
# ============================================================
# Section 8.6 : Export Results
# ============================================================

valid_df.to_csv(

    "Indian_MOREPOC_ValidSamples.csv",

    index=False

)

missing_df.to_csv(

    "Indian_MOREPOC_MissingScenes.csv",

    index=False

)

print("="*70)
print("Files Saved Successfully")
print("="*70)

print("Valid Samples :", valid_df.shape)

print("Missing Samples :", missing_df.shape)

Files Saved Successfully
Valid Samples : (27, 6)
Missing Samples : (0, 0)


### Interpretation

The availability assessment identified all sampling locations that possess at least one Landsat observation within the ±60-day search window.

Samples containing valid imagery will be used for Landsat scene assignment and reflectance extraction, whereas samples without imagery are retained separately for documentation and future investigation.

Separating valid and missing observations before scene assignment prevents Earth Engine execution errors and provides a transparent record of data availability.

# Section 9 : Assign Nearest Landsat Scene

## Objective

Each valid MOREPOC sampling observation is linked to its nearest available Landsat image.

The following information will be assigned:

- Landsat Scene ID
- Acquisition Date
- Landsat Mission
- Cloud Cover
- Temporal Difference (days)

This produces the final scene-assigned dataset required for spectral reflectance extraction.

In [29]:
# ============================================================
# Section 9.1 : Assign Nearest Landsat Scene
# ============================================================

assigned_fc = valid_fc.map(find_nearest_scene)

print("="*70)
print("Scene Assignment Completed")
print("="*70)

print("Assigned Features :", assigned_fc.size().getInfo())

Scene Assignment Completed
Assigned Features : 27


In [30]:
# ============================================================
# Section 9.2 : FeatureCollection → Pandas
# (Preserve Coordinates)
# ============================================================

features = assigned_fc.getInfo()["features"]

records = []

for feature in features:

    props = feature["properties"]

    coords = feature["geometry"]["coordinates"]

    props["lon"] = coords[0]

    props["lat"] = coords[1]

    records.append(props)

assigned_df = pd.DataFrame(records)

print("="*70)
print("Assigned Dataset")
print("="*70)

print(assigned_df.shape)

display(assigned_df.head())

Assigned Dataset
(27, 13)


,cloud_cover,conc_poc,country,date_difference,image_count,riv_id,sample_date,sample_id,scene_date,scene_id,sensor,lon,lat
0,34,3.9,India,0.837279,19,Brahmaputra,1999-10-25,IND_0001,1999-10-24,1_1_1_LT05_136042_19991024,LANDSAT_5,91.74,26.19
1,34,5.4,India,0.837279,19,Brahmaputra,1999-10-25,IND_0002,1999-10-24,1_1_1_LT05_136042_19991024,LANDSAT_5,91.74,26.19
2,34,5.9,India,0.837279,19,Brahmaputra,1999-10-25,IND_0003,1999-10-24,1_1_1_LT05_136042_19991024,LANDSAT_5,91.74,26.19
3,34,5.3,India,0.837279,19,Brahmaputra,1999-10-25,IND_0004,1999-10-24,1_1_1_LT05_136042_19991024,LANDSAT_5,91.74,26.19
4,34,5.5,India,0.837279,19,Brahmaputra,1999-10-25,IND_0005,1999-10-24,1_1_1_LT05_136042_19991024,LANDSAT_5,91.74,26.19


In [31]:
# ============================================================
# Section 9.3 : Quality Check
# ============================================================

print("="*70)
print("Missing Scene IDs")
print("="*70)

print(assigned_df["scene_id"].isna().sum())

print()

print("="*70)
print("Sensor Distribution")
print("="*70)

print(assigned_df["sensor"].value_counts())

print()

print("="*70)
print("Date Difference Statistics")
print("="*70)

print(assigned_df["date_difference"].describe())

Missing Scene IDs
0

Sensor Distribution
sensor
LANDSAT_5    14
LANDSAT_7    13
Name: count, dtype: int64

Date Difference Statistics
count    27.000000
mean      4.646666
std       7.958572
min       0.192924
25%       0.837279
50%       2.807076
75%       4.178295
max      31.841038
Name: date_difference, dtype: float64


In [32]:
# ============================================================
# Section 9.4 : Sort Dataset
# ============================================================

assigned_df = assigned_df.sort_values(

    ["sample_date", "sample_id"]

).reset_index(drop=True)

display(assigned_df.head())

,cloud_cover,conc_poc,country,date_difference,image_count,riv_id,sample_date,sample_id,scene_date,scene_id,sensor,lon,lat
0,51,2.6,India,31.841038,3,Brahmaputra,1997-03-15,IND_0006,1997-02-11,1_1_1_LT05_137042_19970211,LANDSAT_5,91.74,26.19
1,51,0.1,India,31.841038,3,Brahmaputra,1997-03-15,IND_0007,1997-02-11,1_1_1_LT05_137042_19970211,LANDSAT_5,91.74,26.19
2,34,3.9,India,0.837279,19,Brahmaputra,1999-10-25,IND_0001,1999-10-24,1_1_1_LT05_136042_19991024,LANDSAT_5,91.74,26.19
3,34,5.4,India,0.837279,19,Brahmaputra,1999-10-25,IND_0002,1999-10-24,1_1_1_LT05_136042_19991024,LANDSAT_5,91.74,26.19
4,34,5.9,India,0.837279,19,Brahmaputra,1999-10-25,IND_0003,1999-10-24,1_1_1_LT05_136042_19991024,LANDSAT_5,91.74,26.19


In [33]:
# ============================================================
# Section 9.5 : Export
# ============================================================

assigned_df.to_csv(

    "Indian_MOREPOC_Landsat_Assigned.csv",

    index=False

)

print("="*70)
print("Dataset Exported Successfully")
print("="*70)

print("Shape :", assigned_df.shape)

Dataset Exported Successfully
Shape : (27, 13)


In [34]:
# ============================================================
# Section 9.6 : Save Final Scene-Assigned Dataset
# ============================================================

assigned_df.to_csv(

    "Indian_MOREPOC_SceneAssigned.csv",

    index=False

)

print("="*70)
print("Final Scene-Assigned Dataset Saved")
print("="*70)

print("Shape :", assigned_df.shape)

Final Scene-Assigned Dataset Saved
Shape : (27, 13)


### Interpretation

Every valid MOREPOC observation has now been linked with its nearest available Landsat scene.

The exported dataset contains the essential metadata required for reflectance extraction, including scene identifier, acquisition date, satellite mission, cloud cover, and temporal difference.

This dataset forms the direct input for the next stage, where spectral reflectance values (Blue, Green, Red, NIR, SWIR1, and SWIR2) will be extracted from the assigned Landsat imagery.

In [35]:
# ============================================================
# Section 10.1 : Reflectance Extraction Function
# ============================================================

def extract_reflectance(feature):

    geometry = feature.geometry()

    scene_id = feature.get("scene_id")

    sensor = feature.get("sensor")

    if sensor == "LANDSAT_5":

        image = LS5.filter(
            ee.Filter.eq("system:index", scene_id)
        ).first()

    elif sensor == "LANDSAT_7":

        image = LS7.filter(
            ee.Filter.eq("system:index", scene_id)
        ).first()

    elif sensor == "LANDSAT_8":

        image = LS8.filter(
            ee.Filter.eq("system:index", scene_id)
        ).first()

    else:

        image = LS9.filter(
            ee.Filter.eq("system:index", scene_id)
        ).first()

    image = harmonize(
        mask_clouds(image)
    )

    sample = image.sample(
        region=geometry,
        scale=30,
        geometries=False
    ).first()

    return feature.set(
        sample.toDictionary()
    )

In [36]:
# ============================================================
# Section 10.2 : Batch Extraction
# ============================================================

reflectance_fc = assigned_fc.map(get_image)

print("="*70)
print("Reflectance Extraction Completed")
print("="*70)

print(reflectance_fc.size().getInfo())

NameError: name 'get_image' is not defined

In [ ]:
image = harmonize(mask_clouds(image))

print(type(image))

print(image.bandNames().getInfo())